# 📈 PREP_01 — Feature Engineering Indikator Teknikal
## Tesis: Pemodelan Prediksi Volatilitas Return Saham Sektor Energi LQ45

| | |
|---|---|
| **Mahasiswa** | Ilda Nurida (NRP 6032251068) |
| **Program Studi** | Magister Manajemen Teknologi — SIMT ITS |
| **Tahun** | 2026 |
| **Lokasi File** | `02_preprocessing/preprocessing/` |

---

### 🎯 Tujuan
Melakukan *feature engineering* indikator teknikal sebagai variabel input (X1) untuk model XGBoost dan LSTM.

### 📥 Input
- `03_dataset/processed/volatilitas.csv` — dataset volatilitas + harga + log return
- `03_dataset/raw/harga_saham_raw.csv` — data OHLCV lengkap (untuk volume)

### 📤 Output
| File | Lokasi | Kegunaan |
|------|--------|----------|
| `technical_features.csv` | `03_dataset/processed/` | Input model XGBoost & LSTM |
| `tabel_statistik_fitur_teknikal.csv` | `06_dokumen_tesis/tabel/` | **Tabel 4.X** Tesis |
| `gambar_indikator_teknikal_[EMITEN].png` | `06_dokumen_tesis/gambar/EDA/` | **Gambar 4.X** Tesis |

### 📊 Fitur yang Dihitung
| No | Fitur | Parameter | Keterangan |
|----|-------|-----------|------------|
| 1 | MA5, MA10, MA20 | Window 5, 10, 20 hari | Simple Moving Average |
| 2 | EMA12, EMA26 | Span 12, 26 hari | Exponential Moving Average |
| 3 | RSI | 14 hari | Relative Strength Index |
| 4 | MACD + Signal | EMA12-EMA26, Signal=EMA9 | Moving Average Convergence Divergence |
| 5 | Bollinger Bands | MA20 ± 2σ | Upper, Middle, Lower Band |
| 6 | Volume Change (%) | Perubahan % volume | Momentum volume |
| 7 | Volume MA5 | Window 5 hari | Rata-rata volume bergulir |

---
## ⚙️ CELL 1 — Import Library dan Konfigurasi Path

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family' : 'Times New Roman',
    'font.size'   : 10,
    'figure.dpi'  : 150,
})

# ═══════════════════════════════════════════════════════════
# PATH — DETEKSI OTOMATIS
# ═══════════════════════════════════════════════════════════
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
ROOT = NOTEBOOK_DIR
for _ in range(5):
    if (ROOT / '03_dataset').exists():
        break
    ROOT = ROOT.parent

# ── Path Input
PATH_VOLATILITAS = ROOT / '03_dataset' / 'processed' / 'volatilitas.csv'
PATH_HARGA_RAW   = ROOT / '03_dataset' / 'raw'       / 'harga_saham_raw.csv'

# ── Path Output
DIR_PROCESSED    = ROOT / '03_dataset'     / 'processed'
DIR_TABEL        = ROOT / '06_dokumen_tesis' / 'tabel'
DIR_GAMBAR       = ROOT / '06_dokumen_tesis' / 'gambar' / 'EDA'

for d in [DIR_PROCESSED, DIR_TABEL, DIR_GAMBAR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Konfigurasi penelitian
URUTAN_EMITEN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']

NAMA_PERUSAHAAN = {
    'ADRO': 'Adaro Energy Indonesia Tbk',
    'ADMR': 'Adaro Minerals Indonesia Tbk',
    'ITMG': 'Indo Tambangraya Megah Tbk',
    'PTBA': 'Bukit Asam Tbk',
    'MEDC': 'Medco Energi Internasional Tbk',
    'AKRA': 'AKR Corporindo Tbk',
    'PGAS': 'Perusahaan Gas Negara Tbk',
}

WARNA_EMITEN = {
    'ADRO':'#1f77b4','ADMR':'#2ca02c','ITMG':'#ff7f0e',
    'PTBA':'#d62728','MEDC':'#9467bd','AKRA':'#8c564b','PGAS':'#e377c2',
}

# Tambahkan ke konfigurasi Cell 1
KOLOM_EMITEN_KANDIDAT = [
    'Emiten', 'Kode_Emiten', 'kode_emiten',
    'Ticker', 'Symbol', 'EMITEN', 'CODE'
]

print('=' * 65)
print('  ✅ Library dan konfigurasi berhasil dimuat')
print('=' * 65)
print(f'  ROOT project      : {ROOT}')
print(f'  Input volatilitas : {PATH_VOLATILITAS}')
print(f'  Input harga raw   : {PATH_HARGA_RAW}')
print(f'  Output processed  : {DIR_PROCESSED}')
print(f'  Output tabel      : {DIR_TABEL}')
print(f'  Output gambar     : {DIR_GAMBAR}')
print()
print(f'  Emiten penelitian : {URUTAN_EMITEN}')

---
## 📂 CELL 2 — Load Dataset

Load dua dataset:
1. **`volatilitas.csv`** — berisi data harga, log return, dan volatilitas per emiten
2. **`harga_saham_raw.csv`** — berisi data OHLCV lengkap (diperlukan untuk volume)

In [ ]:
print('=' * 65)
print('  📂 CELL 2 — LOAD DAN STANDARISASI DATA HARGA SAHAM')
print('=' * 65)

from pathlib import Path

# ── Path file harga saham
# Prioritas: cari dari folder scraping dulu, lalu folder raw
KANDIDAT_PATH_HARGA = [
    ROOT / '03_dataset' / 'raw' / 'harga_saham_raw.csv',
    Path(r'E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
         r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026'
         r'\01_scraping_data\harga_saham\harga_saham_raw.csv'),
    Path(r'E:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
         r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026'
         r'\03_dataset\raw\harga_saham_raw.csv'),
]

PATH_HARGA_LOAD = None
for p in KANDIDAT_PATH_HARGA:
    if p.exists():
        PATH_HARGA_LOAD = p
        break

if PATH_HARGA_LOAD is None:
    print('❌ File harga saham tidak ditemukan di semua lokasi:')
    for p in KANDIDAT_PATH_HARGA:
        print(f'   {p}')
    raise FileNotFoundError('harga_saham_raw.csv tidak ditemukan')

print(f'  ✅ File ditemukan: {PATH_HARGA_LOAD}')

# ── Load CSV
df_harga_raw = pd.read_csv(PATH_HARGA_LOAD, encoding='utf-8-sig')
print(f'  Shape awal   : {df_harga_raw.shape}')
print(f'  Kolom awal   : {list(df_harga_raw.columns)}')
print()

# ════════════════════════════════════════════════════════════
# DETEKSI DAN STANDARISASI FORMAT
# ════════════════════════════════════════════════════════════

KOLOM_EMITEN_KANDIDAT = [
    'Kode_Emiten', 'kode_emiten', 'KODE_EMITEN',
    'Emiten', 'emiten', 'EMITEN',
    'Ticker', 'ticker', 'TICKER',
    'Symbol', 'symbol', 'SYMBOL',
    'Code', 'code', 'CODE',
    'Kode Emiten', 'KODE EMITEN',
]

kolom          = [str(c) for c in df_harga_raw.columns]
ada_emiten_col = any(c in KOLOM_EMITEN_KANDIDAT for c in kolom)
ada_kode_col   = any(e in kolom for e in URUTAN_EMITEN)

# ════════════════════════════════════════════════════════════
# FORMAT LONG — ada kolom identitas emiten
# ════════════════════════════════════════════════════════════
if ada_emiten_col:
    print('  📋 Format terdeteksi: LONG')
    df_harga = df_harga_raw.copy()

    # ── Rename kolom emiten → 'Emiten'
    for c in df_harga.columns:
        if str(c) in KOLOM_EMITEN_KANDIDAT:
            if str(c) != 'Emiten':
                df_harga.rename(columns={c: 'Emiten'}, inplace=True)
                print(f'  ✅ Rename "{c}" → "Emiten"')
            break

    # ── Rename kolom tanggal → 'Date'
    for c in df_harga.columns:
        if str(c).lower() in ['tanggal','date','tgl','waktu','time']:
            if str(c) != 'Date':
                df_harga.rename(columns={c: 'Date'}, inplace=True)
                print(f'  ✅ Rename "{c}" → "Date"')
            break

    # ── Tentukan kolom harga: Adj_Close > Adj Close > Close
    if 'Adj_Close' in df_harga.columns:
        COL_HARGA = 'Adj_Close'
        print(f'  ✅ Kolom harga: Adj_Close')
    elif 'Adj Close' in df_harga.columns:
        df_harga.rename(columns={'Adj Close': 'Adj_Close'}, inplace=True)
        COL_HARGA = 'Adj_Close'
        print(f'  ✅ Kolom harga: "Adj Close" → "Adj_Close"')
    elif 'Close' in df_harga.columns:
        COL_HARGA = 'Close'
        print(f'  ✅ Kolom harga: Close')
    else:
        close_kandidat = [c for c in df_harga.columns
                          if 'close' in str(c).lower()]
        if close_kandidat:
            COL_HARGA = close_kandidat[0]
            print(f'  ✅ Kolom harga: {COL_HARGA}')
        else:
            raise ValueError(
                f'❌ Kolom harga tidak ditemukan!\n'
                f'   Kolom tersedia: {list(df_harga.columns)}'
            )

    # ── Rename OHLCV ke standar
    rename_ohlcv = {}
    for c in df_harga.columns:
        cl = str(c).lower()
        if cl == 'high'    and c != 'High'  : rename_ohlcv[c] = 'High'
        if cl == 'low'     and c != 'Low'   : rename_ohlcv[c] = 'Low'
        if cl == 'open'    and c != 'Open'  : rename_ohlcv[c] = 'Open'
        if cl == 'volume'  and c != 'Volume': rename_ohlcv[c] = 'Volume'
    if rename_ohlcv:
        df_harga.rename(columns=rename_ohlcv, inplace=True)
        print(f'  ✅ Rename OHLCV: {rename_ohlcv}')

# ════════════════════════════════════════════════════════════
# FORMAT WIDE — kolom = nama emiten langsung
# ════════════════════════════════════════════════════════════
elif ada_kode_col:
    print('  📋 Format terdeteksi: WIDE → konversi ke LONG')

    col_date = next(
        (c for c in kolom
         if str(c).lower() in ['date','tanggal','tgl','waktu']),
        kolom[0]
    )
    emiten_cols = [e for e in URUTAN_EMITEN if e in kolom]
    print(f'  Kolom tanggal    : {col_date}')
    print(f'  Emiten ditemukan : {emiten_cols}')

    df_harga = pd.melt(
        df_harga_raw,
        id_vars    = [col_date],
        value_vars = emiten_cols,
        var_name   = 'Emiten',
        value_name = 'Close',
    )
    df_harga.rename(columns={col_date: 'Date'}, inplace=True)
    COL_HARGA = 'Close'
    print(f'  ✅ Konversi WIDE → LONG selesai: {len(emiten_cols)} emiten')

# ════════════════════════════════════════════════════════════
# FORMAT TIDAK DIKENALI
# ════════════════════════════════════════════════════════════
else:
    print('  ❌ Format tidak dikenali!')
    print(f'  Kolom tersedia : {kolom}')
    raise ValueError('Format kolom tidak dikenali')

# ════════════════════════════════════════════════════════════
# STANDARISASI AKHIR
# ════════════════════════════════════════════════════════════

# ── Konversi Date
df_harga['Date'] = pd.to_datetime(df_harga['Date'], errors='coerce')
n_nat = df_harga['Date'].isna().sum()
if n_nat > 0:
    print(f'  ⚠️  {n_nat} baris tanggal tidak valid, dibuang')
df_harga.dropna(subset=['Date'], inplace=True)

# ── Konversi kolom harga ke numerik
df_harga[COL_HARGA] = pd.to_numeric(df_harga[COL_HARGA], errors='coerce')

# ── Urutkan kronologis per emiten
df_harga.sort_values(['Emiten', 'Date'], inplace=True)
df_harga.reset_index(drop=True, inplace=True)

# ── Simpan COL_HARGA untuk digunakan Cell 3
print(f'\n  📌 COL_HARGA untuk Cell 3 : "{COL_HARGA}"')

# ── Info final
print()
print('=' * 65)
print('  ✅ Data harga siap')
print(f'  Total baris  : {len(df_harga):,}')
print(f'  Kolom        : {list(df_harga.columns)}')
print(f'  Kolom harga  : {COL_HARGA}')
print(f'  Emiten       : {sorted(df_harga["Emiten"].unique().tolist())}')
print(f'  Periode      : {df_harga["Date"].min().date()} → '
      f'{df_harga["Date"].max().date()}')
print()
print('  Preview 5 baris:')
display(df_harga.head(5))

---
## 📊 CELL 3 — Perhitungan Moving Average (MA5, MA10, MA20) dan EMA (EMA12, EMA26)

**Simple Moving Average (SMA):**
$$MA_n = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i}$$

**Exponential Moving Average (EMA):**
$$EMA_t = P_t \cdot \alpha + EMA_{t-1} \cdot (1 - \alpha), \quad \alpha = \frac{2}{n+1}$$

In [ ]:
print('=' * 65)
print('  📊 CELL 3 — HITUNG SEMUA INDIKATOR TEKNIKAL')
print('=' * 65)

def hitung_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)

    avg_gain = gain.ewm(
        alpha=1/window,
        min_periods=window,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/window,
        min_periods=window,
        adjust=False
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


dict_fitur = {}

# ============================================================
# PILIH KOLOM HARGA
# ============================================================
if 'Adj_Close' in df_harga.columns:
    COL_HARGA = 'Adj_Close'
elif 'Adj Close' in df_harga.columns:
    COL_HARGA = 'Adj Close'
else:
    COL_HARGA = 'Close'

print(f'Kolom harga digunakan : {COL_HARGA}')
print()

for emiten in URUTAN_EMITEN:

    df_e = df_harga[df_harga['Emiten'] == emiten].copy()

    if df_e.empty:
        print(f'❌ {emiten}: data kosong')
        continue

    df_e = df_e.sort_values('Date').reset_index(drop=True)

    # ============================================================
    # PILIH KOLOM
    # ============================================================
    kolom_pilih = {
        'Date': 'Date',
        'Emiten': 'Emiten',
        COL_HARGA: 'Close',
        'High': 'High',
        'Low': 'Low',
        'Volume': 'Volume'
    }

    kolom_ada = {
        k: v for k, v in kolom_pilih.items()
        if k in df_e.columns
    }

    df_e = df_e[list(kolom_ada.keys())].copy()
    df_e.rename(columns=kolom_ada, inplace=True)
    df_e = df_e.loc[:, ~df_e.columns.duplicated()]

    if 'High' not in df_e.columns:
        df_e['High'] = df_e['Close']

    if 'Low' not in df_e.columns:
        df_e['Low'] = df_e['Close']

    if 'Volume' not in df_e.columns:
        df_e['Volume'] = 0

    df_e.dropna(subset=['Close'], inplace=True)
    df_e.reset_index(drop=True, inplace=True)

    # ============================================================
    # 1. LOG RETURN
    # ============================================================
    df_e['Log_Return'] = np.log(
        df_e['Close'] / df_e['Close'].shift(1)
    )

    # ============================================================
    # 2. VOLATILITAS
    # ============================================================
    df_e['Volatilitas'] = (
        df_e['Log_Return']
        .rolling(window=20, min_periods=5)
        .std()
    )

    # ============================================================
    # 2B. FITUR AUTOREGRESIF VOLATILITAS (BARU)
    # ============================================================
    df_e['Volatilitas_Return'] = df_e['Volatilitas']

    # ── REVISI (Reviewer 1 / audit baseline GARCH) ───────────────────
    # Target model = Volatilitas_Target = sigma_{t+1}. Lag harus RELATIF
    # TERHADAP TARGET, sehingga Lag1 = sigma_t (sudah diketahui saat
    # penutupan bursa hari t → TIDAK ada leakage).
    # Versi lama: Vol_Lag1 = sigma_{t-1} → model tidak melihat sigma_t.
    df_e['Vol_Lag1'] = df_e['Volatilitas']            # sigma_t
    df_e['Vol_Lag2'] = df_e['Volatilitas'].shift(1)   # sigma_{t-1}
    df_e['Vol_Lag3'] = df_e['Volatilitas'].shift(2)   # sigma_{t-2}

    # Return yang KELUAR dari window 20 hari besok (r_{t-19}).
    # sigma_{t+1} = f(r_{t-18..t}, r_{t+1}) → r_{t-19} keluar dari window.
    df_e['R_Exit'] = df_e['Log_Return'].shift(19)

    df_e['Vol_MA5'] = (
        df_e['Volatilitas']
        .rolling(window=5, min_periods=1)
        .mean()
    )

    df_e['Vol_STD5'] = (
        df_e['Volatilitas']
        .rolling(window=5, min_periods=2)
        .std()
    )

    # ============================================================
    # 3. MOVING AVERAGE
    # ============================================================
    df_e['MA5'] = df_e['Close'].rolling(5, min_periods=1).mean()
    df_e['MA10'] = df_e['Close'].rolling(10, min_periods=1).mean()
    df_e['MA20'] = df_e['Close'].rolling(20, min_periods=1).mean()

    # ============================================================
    # 4. EMA
    # ============================================================
    df_e['EMA12'] = df_e['Close'].ewm(span=12, adjust=False).mean()
    df_e['EMA26'] = df_e['Close'].ewm(span=26, adjust=False).mean()

    # ============================================================
    # 5. RSI
    # ============================================================
    df_e['RSI14'] = hitung_rsi(df_e['Close'], 14)

    # ============================================================
    # 6. MACD
    # ============================================================
    df_e['MACD'] = df_e['EMA12'] - df_e['EMA26']
    df_e['MACD_Signal'] = df_e['MACD'].ewm(span=9, adjust=False).mean()
    df_e['MACD_Hist'] = df_e['MACD'] - df_e['MACD_Signal']

    # ============================================================
    # 7. BOLLINGER BAND
    # ============================================================
    bb_mean = df_e['Close'].rolling(20, min_periods=1).mean()
    bb_std = df_e['Close'].rolling(20, min_periods=1).std()

    df_e['BB_Upper'] = bb_mean + (2 * bb_std)
    df_e['BB_Middle'] = bb_mean
    df_e['BB_Lower'] = bb_mean - (2 * bb_std)

    df_e['BB_Width'] = (
        (df_e['BB_Upper'] - df_e['BB_Lower']) /
        df_e['BB_Middle'].replace(0, np.nan)
    )

    df_e['BB_PctB'] = (
        (df_e['Close'] - df_e['BB_Lower']) /
        (df_e['BB_Upper'] - df_e['BB_Lower']).replace(0, np.nan)
    )

    # ============================================================
    # 8. STOCHASTIC
    # ============================================================
    low14 = df_e['Low'].rolling(14, min_periods=1).min()
    high14 = df_e['High'].rolling(14, min_periods=1).max()

    df_e['Stoch_K'] = (
        (df_e['Close'] - low14) /
        (high14 - low14).replace(0, np.nan) * 100
    )

    df_e['Stoch_D'] = df_e['Stoch_K'].rolling(3, min_periods=1).mean()

    # ============================================================
    # 9. VOLUME FEATURE
    # ============================================================
    df_e['Volume_Change'] = df_e['Volume'].pct_change() * 100
    df_e['Volume_MA5'] = df_e['Volume'].rolling(5, min_periods=1).mean()

    dict_fitur[emiten] = df_e

    print(f'✅ {emiten:<5}: {len(df_e):,} baris')

print()
print(f'✅ SELESAI: {len(dict_fitur)}/{len(URUTAN_EMITEN)} emiten')

In [ ]:
print(dict_fitur['ADRO'].columns.tolist())

---
## 📊 CELL 4 — Perhitungan RSI (Relative Strength Index)

$$RSI = 100 - \frac{100}{1 + RS}, \quad RS = \frac{\text{Rata-rata Kenaikan (14 hari)}}{\text{Rata-rata Penurunan (14 hari)}}$$

- RSI > 70 → overbought (potensi koreksi turun)
- RSI < 30 → oversold (potensi rebound naik)

In [ ]:
print('=' * 65)
print('  📊 CELL 4 — RSI (14 hari)')
print('=' * 65)

def hitung_rsi(series, window=14):
    """
    Menghitung RSI menggunakan metode Wilder's Smoothing
    """

    # Selisih harga
    delta = series.diff()

    # Gain dan Loss
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    # Wilder smoothing
    avg_gain = gain.ewm(
        alpha=1/window,
        min_periods=window,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/window,
        min_periods=window,
        adjust=False
    ).mean()

    # Relative Strength
    rs = avg_gain / avg_loss

    # Hitung RSI
    rsi = 100 - (100 / (1 + rs))

    # Handle edge cases
    rsi = rsi.where(avg_loss != 0, 100)
    rsi = rsi.where(avg_gain != 0, 0)

    # Pastikan range valid
    rsi = rsi.clip(0, 100)

    return rsi


for emiten in URUTAN_EMITEN:
    if emiten not in dict_fitur:
        continue

    df_e = dict_fitur[emiten].copy()

    # Lebih disarankan pakai Adj Close
    price_col = 'Adj Close' if 'Adj Close' in df_e.columns else 'Close'

    df_e['RSI14'] = hitung_rsi(
        df_e[price_col],
        window=14
    )

    dict_fitur[emiten] = df_e

    rsi_mean = df_e['RSI14'].mean()
    rsi_min = df_e['RSI14'].min()
    rsi_max = df_e['RSI14'].max()
    missing = df_e['RSI14'].isna().sum()

    print(
        f'  ✅ {emiten}: '
        f'RSI14 | mean={rsi_mean:.2f}, '
        f'min={rsi_min:.2f}, '
        f'max={rsi_max:.2f}, '
        f'missing={missing}'
    )

print()
print('  ✅ Perhitungan RSI selesai untuk semua emiten')

---
## 📊 CELL 5 — Perhitungan MACD dan Signal Line

$$MACD = EMA_{12} - EMA_{26}$$
$$Signal = EMA_9(MACD)$$
$$Histogram = MACD - Signal$$

- MACD > Signal → sinyal bullish (beli)
- MACD < Signal → sinyal bearish (jual)

In [ ]:
print('=' * 65)
print('  📊 CELL 5 — MACD + Signal Line')
print('=' * 65)

for emiten in URUTAN_EMITEN:
    if emiten not in dict_fitur:
        continue

    df_e = dict_fitur[emiten].copy()  # 🔥 penting: hindari view error

    # ── pastikan data sudah urut (WAJIB untuk EMA)
    df_e = df_e.sort_values('Date')

    # ── MACD = EMA12 - EMA26
    df_e['MACD'] = df_e['EMA12'] - df_e['EMA26']

    # ── Signal Line = EMA9 dari MACD
    df_e['MACD_Signal'] = df_e['MACD'].ewm(span=9, adjust=False).mean()

    # ── Histogram
    df_e['MACD_Hist'] = df_e['MACD'] - df_e['MACD_Signal']

    # ── simpan kembali
    dict_fitur[emiten] = df_e

    print(f'  ✅ {emiten}: MACD, Signal, Histogram selesai')

print()
print('  ✅ MACD selesai untuk semua emiten')

---
## 📊 CELL 6 — Perhitungan Bollinger Bands

$$BB_{Upper} = MA_{20} + 2\sigma_{20}$$
$$BB_{Middle} = MA_{20}$$
$$BB_{Lower} = MA_{20} - 2\sigma_{20}$$
$$BB_{Width} = \frac{BB_{Upper} - BB_{Lower}}{BB_{Middle}}$$

- Harga mendekati Upper Band → potensi overbought
- Harga mendekati Lower Band → potensi oversold
- BB Width sempit → volatilitas rendah (breakout potential)

In [ ]:
print('=' * 65)
print('  📊 CELL 6 — Bollinger Bands (window=20, std=2)')
print('=' * 65)

WINDOW = 20
NUM_STD = 2

for emiten in URUTAN_EMITEN:
    if emiten not in dict_fitur:
        continue

    df_e = dict_fitur[emiten].copy()

    # Gunakan Adj Close jika tersedia
    price_col = 'Adj Close' if 'Adj Close' in df_e.columns else 'Close'

    # ========================================================
    # Rolling Mean & Std
    # ========================================================
    rolling_mean = (
        df_e[price_col]
        .rolling(window=WINDOW, min_periods=WINDOW)
        .mean()
    )

    rolling_std = (
        df_e[price_col]
        .rolling(window=WINDOW, min_periods=WINDOW)
        .std()
    )

    # ========================================================
    # Bollinger Bands
    # ========================================================
    df_e['BB_Middle'] = rolling_mean
    df_e['BB_Upper'] = rolling_mean + (NUM_STD * rolling_std)
    df_e['BB_Lower'] = rolling_mean - (NUM_STD * rolling_std)

    # ========================================================
    # Bollinger Band Width
    # ========================================================
    df_e['BB_Width'] = (
        (df_e['BB_Upper'] - df_e['BB_Lower']) /
        df_e['BB_Middle'].replace(0, np.nan)
    )

    # ========================================================
    # Percent B
    # ========================================================
    band_range = (
        df_e['BB_Upper'] - df_e['BB_Lower']
    ).replace(0, np.nan)

    df_e['BB_PctB'] = (
        (df_e[price_col] - df_e['BB_Lower']) /
        band_range
    )

    dict_fitur[emiten] = df_e

    bb_mean = df_e['BB_Width'].mean()
    bb_max = df_e['BB_Width'].max()
    missing = df_e['BB_Width'].isna().sum()

    print(
        f'  ✅ {emiten}: '
        f'BB_Width mean={bb_mean:.4f}, '
        f'max={bb_max:.4f}, '
        f'missing={missing}'
    )

print()
print('  ✅ Perhitungan Bollinger Bands selesai untuk semua emiten')

---
## 📊 CELL 7 — Perhitungan Fitur Volume

$$Volume\_Change = \frac{V_t - V_{t-1}}{V_{t-1}} \times 100\%$$
$$Volume\_MA5 = \frac{1}{5} \sum_{i=0}^{4} V_{t-i}$$

In [ ]:
# ============================================================
# CELL 7 — FITUR VOLUME
# ============================================================

print('=' * 70)
print('  📊 CELL 7 — Fitur Volume')
print('=' * 70)

for emiten in URUTAN_EMITEN:

    if emiten not in dict_fitur:
        continue

    df_e = dict_fitur[emiten].copy()

    # --------------------------------------------------------
    # Validasi kolom volume
    # --------------------------------------------------------
    if 'Volume' not in df_e.columns:
        df_e['Volume'] = 0
        print(f'⚠️  {emiten}: Volume tidak tersedia → diisi 0')
        dict_fitur[emiten] = df_e
        continue

    # ========================================================
    # 1. Volume Change (%)
    # ========================================================
    df_e['Volume_Change'] = (
        df_e['Volume'].pct_change() * 100
    )

    # ========================================================
    # 2. Moving Average Volume (5 hari)
    # ========================================================
    df_e['Volume_MA5'] = (
        df_e['Volume']
        .rolling(window=5, min_periods=5)
        .mean()
    )

    # ========================================================
    # 3. Volume Z-Score
    # ========================================================
    vol_mean = df_e['Volume'].rolling(
        window=20,
        min_periods=20
    ).mean()

    vol_std = df_e['Volume'].rolling(
        window=20,
        min_periods=20
    ).std()

    df_e['Volume_ZScore'] = (
        (df_e['Volume'] - vol_mean) /
        vol_std.replace(0, np.nan)
    )

    dict_fitur[emiten] = df_e

    mean_vol = df_e['Volume'].mean()
    missing = df_e['Volume_MA5'].isna().sum()

    print(
        f'✅ {emiten:<6}: '
        f'Mean Volume={mean_vol:,.0f} | '
        f'Missing={missing}'
    )

print()
print('✅ Perhitungan fitur volume selesai untuk semua emiten')

---
## 📉 CELL 7B — Fitur Hybrid: GARCH(1,1) Forecast (REVISI Reviewer 1)

`GARCH_Forecast` = prediksi volatilitas rolling 20 hari untuk **t+1** dari model GARCH(1,1)-t (Bollerslev, 1986), dipakai sebagai **fitur** XGBoost & LSTM (pendekatan hybrid GARCH–ML).

**Anti-leakage:** untuk setiap baris $t$, parameter GARCH diestimasi hanya dari data $\le t$ (expanding window, re-estimasi tiap 20 hari bursa). Untuk 250 hari pertama (*burn-in*) dipakai parameter dari estimasi 250 hari pertama; seluruh baris ini berada jauh di dalam data latih, sehingga tidak ada informasi data uji yang masuk.

Konversi varians harian GARCH ke target rolling 20 hari (19 return diketahui + 1 return prediksi):
$$E[S^2] = \frac{1}{19}\left[\sum_{k=1}^{19} x_k^2 + \mu^2 + \hat\sigma^2_{t+1} - 20\left(\frac{\sum x_k + \mu}{20}\right)^2 - \frac{\hat\sigma^2_{t+1}}{20}\right],\quad \text{GARCH\_Forecast}_t = \sqrt{E[S^2]}$$


In [ ]:
# ============================================================
# CELL 7B — FITUR HYBRID: GARCH_Forecast (pip install arch)
# ============================================================
from arch import arch_model

WINDOW_VOL  = 20    # sama dengan window Volatilitas
BURN_IN     = 250   # minimal observasi untuk estimasi GARCH pertama
REFIT_EVERY = 20    # re-estimasi tiap 20 hari bursa
SCALE       = 100.0

def expected_rolling_std(r_known, m, s2, ddof=1):
    n = len(r_known) + 1
    sx, sx2 = r_known.sum(), (r_known ** 2).sum()
    var = (sx2 + m**2 + s2 - n * (((sx + m) / n) ** 2 + s2 / n**2)) / (n - ddof)
    return np.sqrt(max(var, 1e-12))

def garch_forecast_feature(log_ret):
    s = log_ret.dropna()
    labels, r_vals = s.index, s.values
    r = pd.Series(r_vals * SCALE)            # indeks posisi 0..n-1
    n = len(r)
    out = pd.Series(np.nan, index=log_ret.index)
    if n < BURN_IN + WINDOW_VOL:
        return out
    starts = list(range(BURN_IN - 1, n, REFIT_EVERY))
    for k, st in enumerate(starts):
        first = 0 if k == 0 else st
        end   = starts[k + 1] if k + 1 < len(starts) else n
        last_obs = st + 1 if st + 1 < n else None          # estimasi pakai data s.d. posisi st
        res = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t') \
                .fit(last_obs=last_obs, disp='off', show_warning=False)
        var_next = res.forecast(start=first, horizon=1, reindex=False).variance['h.1']
        mu = res.params['mu'] / SCALE
        for p in range(max(first, WINDOW_VOL - 2), end):
            s2 = var_next.loc[p] / SCALE**2
            out[labels[p]] = expected_rolling_std(r_vals[p - (WINDOW_VOL - 2): p + 1], mu, s2)
    return out

print('=' * 70); print('  📉 CELL 7B — GARCH_Forecast (expanding window, tanpa leakage)'); print('=' * 70)
for emiten in URUTAN_EMITEN:
    if emiten not in dict_fitur:
        continue
    df_e = dict_fitur[emiten].sort_values('Date').reset_index(drop=True)
    df_e['GARCH_Forecast'] = garch_forecast_feature(df_e['Log_Return'])
    dict_fitur[emiten] = df_e
    print(f'  ✅ {emiten:<5}: GARCH_Forecast terisi {df_e["GARCH_Forecast"].notna().sum():,} '
          f'/ {len(df_e):,} baris')


In [ ]:
# ============================================================
# CELL 7C — VALIDASI FITUR REVISI (wajib lolos)
# ============================================================
rows = []
for emiten in URUTAN_EMITEN:
    d = dict_fitur[emiten]
    sig_next = d['Log_Return'].rolling(20).std().shift(-1)   # = Volatilitas_Target
    m = d['GARCH_Forecast'].notna() & sig_next.notna()
    err = (d.loc[m, 'GARCH_Forecast'] - sig_next[m]).abs() / sig_next[m]
    rows.append({
        'Emiten': emiten,
        'Vol_Lag1 = sigma_t': bool(np.allclose(d['Vol_Lag1'], d['Volatilitas'], equal_nan=True)),
        'R_Exit = r_(t-19)' : bool(np.allclose(d['R_Exit'], d['Log_Return'].shift(19), equal_nan=True)),
        'Corr GARCH vs sigma_(t+1)': round(d.loc[m, 'GARCH_Forecast'].corr(sig_next[m]), 4),
        'MAPE GARCH (%)': round(err.mean() * 100, 2),
    })
print(pd.DataFrame(rows).to_string(index=False))


---
## 🔍 CELL 8 — Cek Missing Values dan Gabungkan Dataset

In [ ]:
# ============================================================
# CELL 8 — CEK MISSING VALUES & GABUNGKAN DATASET
# ============================================================

print('=' * 70)
print('  🔍 CELL 8 — CEK MISSING VALUES & GABUNGKAN DATASET')
print('=' * 70)

# ------------------------------------------------------------
# Gabungkan seluruh emiten
# ------------------------------------------------------------
list_gabung = []

for emiten in URUTAN_EMITEN:
    if emiten not in dict_fitur:
        continue

    df_e = dict_fitur[emiten].copy()
    list_gabung.append(df_e)

df_all = pd.concat(list_gabung, ignore_index=True)

# Urut kronologis
df_all = df_all.sort_values(
    ['Emiten', 'Date']
).reset_index(drop=True)

print(f'Total baris : {len(df_all):,}')
print(f'Total kolom : {len(df_all.columns)}')
print()

# ------------------------------------------------------------
# Missing Values Summary
# ------------------------------------------------------------
mv = df_all.isna().sum()
mv_pct = (mv / len(df_all) * 100).round(2)

df_mv = pd.DataFrame({
    'Missing': mv,
    'Persen (%)': mv_pct
})

df_mv = df_mv[df_mv['Missing'] > 0]

print('Missing values per kolom:')
print()

if df_mv.empty:
    print('✅ Tidak ada missing values')
else:
    display(df_mv.sort_values('Missing', ascending=False))

# ------------------------------------------------------------
# Missing per Emiten
# ------------------------------------------------------------
print()
print('Missing values per emiten:')

for emiten in URUTAN_EMITEN:
    sub = df_all[df_all['Emiten'] == emiten]
    missing_total = sub.isna().sum().sum()

    print(f'  {emiten:<6}: {missing_total:,}')

# ------------------------------------------------------------
# Drop rows awal akibat rolling window
# ------------------------------------------------------------
rows_before = len(df_all)

kolom_kritis = [
    'Volatilitas',
    'MA20',
    'RSI14',
    'BB_Width'
]

kolom_kritis = [
    c for c in kolom_kritis
    if c in df_all.columns
]

df_all = df_all.dropna(
    subset=kolom_kritis
).reset_index(drop=True)

rows_after = len(df_all)
rows_dropped = rows_before - rows_after

print()
print(f'Rows sebelum cleaning : {rows_before:,}')
print(f'Rows sesudah cleaning : {rows_after:,}')
print(f'Rows terhapus         : {rows_dropped:,}')

print()
print('Preview dataset final:')
display(df_all.head())

---
## 📊 CELL 9 — Statistik Deskriptif Fitur Teknikal (→ Tabel 4.X Tesis)

In [ ]:
print('=' * 65)
print('  📋 STATISTIK DESKRIPTIF FITUR TEKNIKAL — Tabel 4.X')
print('=' * 65)

KOLOM_FITUR = ['Close','MA5','MA10','MA20','EMA12','EMA26',
               'RSI14','MACD','MACD_Signal','MACD_Hist',
               'BB_Upper','BB_Lower','BB_Width',
               'Volume_Change','Volume_MA5']

kolom_ada = [c for c in KOLOM_FITUR if c in df_all.columns]

# ── Statistik per emiten
list_stat = []
for emiten in URUTAN_EMITEN:
    df_e = df_all[df_all['Emiten'] == emiten][kolom_ada]
    stat = df_e.describe().T
    stat['Skewness'] = df_e.skew()
    stat['Kurtosis'] = df_e.kurtosis()
    stat['Emiten']   = emiten
    stat = stat.reset_index().rename(columns={'index': 'Fitur'})
    list_stat.append(stat)

df_stat = pd.concat(list_stat, ignore_index=True)
df_stat = df_stat[['Emiten','Fitur','mean','std','min','50%','max','Skewness','Kurtosis']]
df_stat.columns = ['Emiten','Fitur','Mean','Std','Min','Median','Max','Skewness','Kurtosis']
df_stat = df_stat.round(4)

print(df_stat.to_string(index=False))

# ── Simpan
path_stat = DIR_TABEL / 'tabel_statistik_fitur_teknikal.csv'
df_stat.to_csv(path_stat, index=False, encoding='utf-8-sig')
print(f'\n  💾 Tabel disimpan: {path_stat}')

---
## 📊 CELL 10 — Visualisasi Indikator Teknikal per Emiten (→ Gambar 4.X Tesis)

Setiap emiten ditampilkan dalam satu gambar dengan 4 panel:
1. Harga Close + MA + Bollinger Bands
2. RSI (14)
3. MACD + Signal
4. Volume + Volume MA5

In [ ]:
print('=' * 65)
print('  📊 VISUALISASI INDIKATOR TEKNIKAL PER EMITEN')
print('=' * 65)

for emiten in URUTAN_EMITEN:
    df_e = df_all[df_all['Emiten'] == emiten].copy()
    if df_e.empty:
        print(f'  ⚠️  {emiten}: Data kosong, skip')
        continue

    warna = WARNA_EMITEN.get(emiten, '#333333')
    nama  = NAMA_PERUSAHAAN.get(emiten, emiten)

    fig = plt.figure(figsize=(16, 14))
    fig.suptitle(
        f'Indikator Teknikal — {emiten} ({nama})\n'
        f'Periode Januari 2022 – Juni 2026',
        fontsize=13, fontweight='bold', y=0.98
    )

    gs = gridspec.GridSpec(4, 1, figure=fig,
                           height_ratios=[3, 1.2, 1.2, 1.2],
                           hspace=0.08)

    # ── Panel 1: Harga + MA + Bollinger Bands
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(df_e['Date'], df_e['Close'],
             color=warna, linewidth=1.2, label='Close', zorder=5)
    if 'MA5' in df_e.columns:
        ax1.plot(df_e['Date'], df_e['MA5'],
                 color='#ff7f0e', linewidth=0.8, linestyle='--', label='MA5', alpha=0.8)
    if 'MA20' in df_e.columns:
        ax1.plot(df_e['Date'], df_e['MA20'],
                 color='#2ca02c', linewidth=0.8, linestyle='--', label='MA20', alpha=0.8)
    if 'BB_Upper' in df_e.columns and 'BB_Lower' in df_e.columns:
        ax1.fill_between(df_e['Date'], df_e['BB_Upper'], df_e['BB_Lower'],
                         alpha=0.1, color=warna, label='Bollinger Band')
        ax1.plot(df_e['Date'], df_e['BB_Upper'],
                 color=warna, linewidth=0.5, linestyle=':', alpha=0.6)
        ax1.plot(df_e['Date'], df_e['BB_Lower'],
                 color=warna, linewidth=0.5, linestyle=':', alpha=0.6)
    ax1.set_ylabel('Harga (Rp)', fontsize=10)
    ax1.legend(loc='upper left', fontsize=8, ncol=4)
    ax1.grid(alpha=0.3, linestyle='--')
    ax1.spines[['top','right']].set_visible(False)
    ax1.tick_params(labelbottom=False)

    # ── Panel 2: RSI
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    if 'RSI14' in df_e.columns:
        ax2.plot(df_e['Date'], df_e['RSI14'],
                 color='#9467bd', linewidth=1.0, label='RSI (14)')
        ax2.axhline(70, color='red',   linewidth=0.8, linestyle='--', alpha=0.7)
        ax2.axhline(30, color='green', linewidth=0.8, linestyle='--', alpha=0.7)
        ax2.axhline(50, color='gray',  linewidth=0.5, linestyle=':', alpha=0.5)
        ax2.fill_between(df_e['Date'], df_e['RSI14'], 70,
                         where=df_e['RSI14'] >= 70, alpha=0.2, color='red')
        ax2.fill_between(df_e['Date'], df_e['RSI14'], 30,
                         where=df_e['RSI14'] <= 30, alpha=0.2, color='green')
        ax2.set_ylim(0, 100)
    ax2.set_ylabel('RSI (14)', fontsize=10)
    ax2.legend(loc='upper left', fontsize=8)
    ax2.grid(alpha=0.3, linestyle='--')
    ax2.spines[['top','right']].set_visible(False)
    ax2.tick_params(labelbottom=False)

    # ── Panel 3: MACD
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    if 'MACD' in df_e.columns and 'MACD_Signal' in df_e.columns:
        ax3.plot(df_e['Date'], df_e['MACD'],
                 color='#1f77b4', linewidth=1.0, label='MACD')
        ax3.plot(df_e['Date'], df_e['MACD_Signal'],
                 color='#d62728', linewidth=1.0, linestyle='--', label='Signal')
        if 'MACD_Hist' in df_e.columns:
            colors_hist = df_e['MACD_Hist'].apply(
                lambda x: '#2ca02c' if x >= 0 else '#d62728')
            ax3.bar(df_e['Date'], df_e['MACD_Hist'],
                    color=colors_hist, alpha=0.4, width=1, label='Histogram')
        ax3.axhline(0, color='gray', linewidth=0.5, linestyle=':')
    ax3.set_ylabel('MACD', fontsize=10)
    ax3.legend(loc='upper left', fontsize=8, ncol=3)
    ax3.grid(alpha=0.3, linestyle='--')
    ax3.spines[['top','right']].set_visible(False)
    ax3.tick_params(labelbottom=False)

    # ── Panel 4: Volume
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    if 'Volume' in df_e.columns and df_e['Volume'].sum() > 0:
        ax4.bar(df_e['Date'], df_e['Volume'],
                color=warna, alpha=0.4, width=1, label='Volume')
        if 'Volume_MA5' in df_e.columns:
            ax4.plot(df_e['Date'], df_e['Volume_MA5'],
                     color='#ff7f0e', linewidth=1.0, label='Volume MA5')
        ax4.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
    else:
        ax4.text(0.5, 0.5, 'Data Volume tidak tersedia',
                 ha='center', va='center', transform=ax4.transAxes,
                 fontsize=10, color='gray')
    ax4.set_ylabel('Volume', fontsize=10)
    ax4.set_xlabel('Tanggal', fontsize=10)
    ax4.legend(loc='upper left', fontsize=8)
    ax4.grid(alpha=0.3, linestyle='--')
    ax4.spines[['top','right']].set_visible(False)

    import matplotlib.dates as mdates
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    path_gambar = DIR_GAMBAR / f'gambar_indikator_teknikal_{emiten}.png'
    plt.savefig(path_gambar, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Gambar disimpan: {path_gambar}')
    plt.close()

In [ ]:
df_all = pd.concat(
    list(dict_fitur.values()),
    ignore_index=True
)

print(df_all.columns.tolist())

---
## 💾 CELL 11 — Simpan Dataset Teknikal Final

In [ ]:
print('=' * 65)
print('  💾 SIMPAN DATASET TEKNIKAL FINAL')
print('=' * 65)

df_save = df_all.copy()
df_save['Date'] = pd.to_datetime(df_save['Date']).dt.strftime('%Y-%m-%d')

path_output = DIR_PROCESSED / 'technical_features.csv'
df_save.to_csv(path_output, index=False, encoding='utf-8-sig')

print(f'SAVE PATH : {path_output.resolve()}')
print(f'Total baris : {len(df_save):,}')
print(f'Total kolom : {len(df_save.columns)}')

# VALIDASI RELOAD
cek = pd.read_csv(path_output)

print('\nKolom hasil reload:')
print(cek.columns.tolist())